In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 【读取步骤】
# 读取 Train/Test application 主表和6张卫星特征表
# ============================================================

# 1. 读取训练集和测试集 application 特征
application_train_features = pd.read_parquet(
    "data/processed/application_train_features.parquet"
)

application_test_features = pd.read_parquet(
    "data/processed/application_test_features.parquet"
)


# 2. 读取6张外部客户级特征表
bureau_features = pd.read_parquet(
    "data/processed/bureau_features.parquet"
)

bureau_balance_features = pd.read_parquet(
    "data/processed/bureau_balance_features.parquet"
)

previous_application_features = pd.read_parquet(
    "data/processed/previous_application_features.parquet"
)

installments_features = pd.read_parquet(
    "data/processed/installments_features.parquet"
)

pos_cash_features = pd.read_parquet(
    "data/processed/pos_cash_features.parquet"
)

credit_card_features = pd.read_parquet(
    "data/processed/credit_card_features.parquet"
)


# 3. 查看所有特征表规模
feature_tables = {
    "application_train": application_train_features,
    "application_test": application_test_features,
    "bureau": bureau_features,
    "bureau_balance": bureau_balance_features,
    "previous_application": previous_application_features,
    "installments": installments_features,
    "pos_cash": pos_cash_features,
    "credit_card": credit_card_features
}

for table_name, df in feature_tables.items():
    print(
        table_name,
        df.shape,
        "重复客户:",
        df["SK_ID_CURR"].duplicated().sum()
    )

application_train (307511, 168) 重复客户: 0
application_test (48744, 167) 重复客户: 0
bureau (305811, 26) 重复客户: 0
bureau_balance (134542, 25) 重复客户: 0
previous_application (338857, 57) 重复客户: 0
installments (339587, 43) 重复客户: 0
pos_cash (337252, 51) 重复客户: 0
credit_card (103558, 81) 重复客户: 0


In [2]:
# ============================================================
# 【合并步骤】
# 分别合并训练集和测试集
# ============================================================


# 1. 定义需要合并的6张卫星表
satellite_tables = [
    ("bureau", bureau_features),
    ("bureau_balance", bureau_balance_features),
    ("previous_application", previous_application_features),
    ("installments", installments_features),
    ("pos_cash", pos_cash_features),
    ("credit_card", credit_card_features)
]


# 2. 以 application_train_features 作为训练集主表
model_train = application_train_features.copy()


# 3. 依次左连接训练集
for table_name, feature_df in satellite_tables:

    before_rows = len(model_train)
    before_columns = model_train.shape[1]

    model_train = model_train.merge(
        feature_df,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )

    print(
        f"Train 合并 {table_name}:",
        f"行数 {before_rows} → {len(model_train)}，",
        f"列数 {before_columns} → {model_train.shape[1]}"
    )


# 4. 以 application_test_features 作为测试集主表
model_test = application_test_features.copy()


# 5. 依次左连接测试集
for table_name, feature_df in satellite_tables:

    before_rows = len(model_test)
    before_columns = model_test.shape[1]

    model_test = model_test.merge(
        feature_df,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )

    print(
        f"Test 合并 {table_name}:",
        f"行数 {before_rows} → {len(model_test)}，",
        f"列数 {before_columns} → {model_test.shape[1]}"
    )

Train 合并 bureau: 行数 307511 → 307511， 列数 168 → 193
Train 合并 bureau_balance: 行数 307511 → 307511， 列数 193 → 217
Train 合并 previous_application: 行数 307511 → 307511， 列数 217 → 273
Train 合并 installments: 行数 307511 → 307511， 列数 273 → 315
Train 合并 pos_cash: 行数 307511 → 307511， 列数 315 → 365
Train 合并 credit_card: 行数 307511 → 307511， 列数 365 → 445
Test 合并 bureau: 行数 48744 → 48744， 列数 167 → 192
Test 合并 bureau_balance: 行数 48744 → 48744， 列数 192 → 216
Test 合并 previous_application: 行数 48744 → 48744， 列数 216 → 272
Test 合并 installments: 行数 48744 → 48744， 列数 272 → 314
Test 合并 pos_cash: 行数 48744 → 48744， 列数 314 → 364
Test 合并 credit_card: 行数 48744 → 48744， 列数 364 → 444


In [3]:
# ============================================================
# 【检查步骤 1】
# 检查最终数据粒度
# ============================================================

print(
    "model_train Shape:",
    model_train.shape
)

print(
    "model_test Shape:",
    model_test.shape
)

print(
    "训练集重复客户数量:",
    model_train["SK_ID_CURR"].duplicated().sum()
)

print(
    "测试集重复客户数量:",
    model_test["SK_ID_CURR"].duplicated().sum()
)

print(
    "训练集包含 TARGET:",
    "TARGET" in model_train.columns
)

print(
    "测试集包含 TARGET:",
    "TARGET" in model_test.columns
)

model_train Shape: (307511, 445)
model_test Shape: (48744, 444)
训练集重复客户数量: 0
测试集重复客户数量: 0
训练集包含 TARGET: True
测试集包含 TARGET: False


In [4]:
# ============================================================
# 【检查步骤 2】
# 检查重复后缀字段
# ============================================================

train_suffix_columns = [
    col
    for col in model_train.columns
    if col.endswith("_x") or col.endswith("_y")
]

test_suffix_columns = [
    col
    for col in model_test.columns
    if col.endswith("_x") or col.endswith("_y")
]


print(
    "训练集重复后缀字段:",
    train_suffix_columns
)

print(
    "测试集重复后缀字段:",
    test_suffix_columns
)

训练集重复后缀字段: []
测试集重复后缀字段: []


In [8]:
# ============================================================
# 【特征工程】
# 建立6张外部表的数据可用性标记
# ============================================================


# 1. 是否存在 bureau 历史记录
model_train["HAS_BUREAU_HISTORY"] = (
    model_train["BUREAU_RECORD_COUNT"]
    .notna()
    .astype("int8")
)

model_test["HAS_BUREAU_HISTORY"] = (
    model_test["BUREAU_RECORD_COUNT"]
    .notna()
    .astype("int8")
)


# 2. 是否存在 bureau_balance 历史记录
model_train["HAS_BUREAU_BALANCE_HISTORY"] = (
    model_train["BB_LOAN_COUNT"]
    .notna()
    .astype("int8")
)

model_test["HAS_BUREAU_BALANCE_HISTORY"] = (
    model_test["BB_LOAN_COUNT"]
    .notna()
    .astype("int8")
)


# 3. 是否存在 previous_application 历史记录
model_train["HAS_PREVIOUS_APPLICATION_HISTORY"] = (
    model_train["PREV_APPLICATION_COUNT"]
    .notna()
    .astype("int8")
)

model_test["HAS_PREVIOUS_APPLICATION_HISTORY"] = (
    model_test["PREV_APPLICATION_COUNT"]
    .notna()
    .astype("int8")
)


# 4. 是否存在 installments 历史记录
model_train["HAS_INSTALLMENT_HISTORY"] = (
    model_train["INSTALMENT_COUNT"]
    .notna()
    .astype("int8")
)

model_test["HAS_INSTALLMENT_HISTORY"] = (
    model_test["INSTALMENT_COUNT"]
    .notna()
    .astype("int8")
)


# 5. 是否存在 POS_CASH 历史记录
model_train["HAS_POS_CASH_HISTORY"] = (
    model_train["POS_LOAN_COUNT"]
    .notna()
    .astype("int8")
)

model_test["HAS_POS_CASH_HISTORY"] = (
    model_test["POS_LOAN_COUNT"]
    .notna()
    .astype("int8")
)


# 6. 是否存在信用卡历史记录
model_train["HAS_CREDIT_CARD_HISTORY"] = (
    model_train["CC_ACCOUNT_COUNT"]
    .notna()
    .astype("int8")
)

model_test["HAS_CREDIT_CARD_HISTORY"] = (
    model_test["CC_ACCOUNT_COUNT"]
    .notna()
    .astype("int8")
)

In [9]:
availability_source_columns = {
    "HAS_BUREAU_HISTORY": "BUREAU_RECORD_COUNT",
    "HAS_BUREAU_BALANCE_HISTORY": "BB_LOAN_COUNT",
    "HAS_PREVIOUS_APPLICATION_HISTORY": "PREV_APPLICATION_COUNT",
    "HAS_INSTALLMENT_HISTORY": "INSTALMENT_COUNT",
    "HAS_POS_CASH_HISTORY": "POS_LOAN_COUNT",
    "HAS_CREDIT_CARD_HISTORY": "CC_ACCOUNT_COUNT"
}

for indicator_col, source_col in availability_source_columns.items():

    model_train[indicator_col] = (
        model_train[source_col]
        .notna()
        .astype("int8")
    )

    model_test[indicator_col] = (
        model_test[source_col]
        .notna()
        .astype("int8")
    )

In [10]:
model_train[
    list(availability_source_columns.keys())
].mean()

HAS_BUREAU_HISTORY                  0.856851
HAS_BUREAU_BALANCE_HISTORY          0.299927
HAS_PREVIOUS_APPLICATION_HISTORY    0.946493
HAS_INSTALLMENT_HISTORY             0.948399
HAS_POS_CASH_HISTORY                0.941248
HAS_CREDIT_CARD_HISTORY             0.282608
dtype: float64

In [11]:
model_test[
    list(availability_source_columns.keys())
].mean()

HAS_BUREAU_HISTORY                  0.868209
HAS_BUREAU_BALANCE_HISTORY          0.868025
HAS_PREVIOUS_APPLICATION_HISTORY    0.980634
HAS_INSTALLMENT_HISTORY             0.983588
HAS_POS_CASH_HISTORY                0.980798
HAS_CREDIT_CARD_HISTORY             0.341642
dtype: float64

In [12]:
model_train_feature_columns = [
    col
    for col in model_train.columns
    if col != "TARGET"
]

print(
    "最终 Train/Test 字段是否一致:",
    model_train_feature_columns
    == model_test.columns.tolist()
)

print(
    "训练集字段数量（不含 TARGET）:",
    len(model_train_feature_columns)
)

print(
    "测试集字段数量:",
    model_test.shape[1]
)

最终 Train/Test 字段是否一致: True
训练集字段数量（不含 TARGET）: 450
测试集字段数量: 450


In [13]:
from pathlib import Path


# 1. 建立输出路径
train_output_path = Path(
    "data/processed/model_train_raw_features.parquet"
)

test_output_path = Path(
    "data/processed/model_test_raw_features.parquet"
)


# 2. 保存训练集
model_train.to_parquet(
    train_output_path,
    index=False
)


# 3. 保存测试集
model_test.to_parquet(
    test_output_path,
    index=False
)


# 4. 检查保存结果
print(
    "训练集文件是否存在:",
    train_output_path.exists()
)

print(
    "测试集文件是否存在:",
    test_output_path.exists()
)

print(
    "model_train Shape:",
    model_train.shape
)

print(
    "model_test Shape:",
    model_test.shape
)

训练集文件是否存在: True
测试集文件是否存在: True
model_train Shape: (307511, 451)
model_test Shape: (48744, 450)
